In [1]:
import sounddevice as sd
print("PortAudio:", sd.get_portaudio_version())
print("Host APIs:", [a["name"] for a in sd.query_hostapis()])
print("Default device:", sd.default.device)
# Is 24k supported on current device?
try:
    sd.check_output_settings(samplerate=24000, channels=1)
    print("24 kHz OK on current device")
except Exception as e:
    print("24 kHz fails on current device:", e)

PortAudio: (1246720, 'PortAudio V19.6.0-devel, revision 396fe4b6699ae929d3a685b3ef8a7e97396139a4')
Host APIs: ['ALSA', 'OSS']
Default device: [4, 0]
24 kHz fails on current device: Invalid sample rate [PaErrorCode -9997]


In [2]:
import sys, json, sounddevice as sd, numpy as np
print("Python:", sys.executable)
print("PortAudio:", sd.get_portaudio_version())

apis = [a["name"] for a in sd.query_hostapis()]
print("Host APIs:", apis)

devs = sd.query_devices()
out_devs = [d for d in devs if d["max_output_channels"] > 0]
print(f"Output devices ({len(out_devs)}):")
for d in out_devs:
    print(f"  [{d['index']:>2}] {d['name']}  (hostapi={apis[d['hostapi']]})  "
          f"max_out={d['max_output_channels']}")
    
print("Default device:", sd.default.device)

def ok_24k(device=None, ch=1):
    try:
        sd.check_output_settings(device=device, samplerate=24000, channels=ch)
        return True
    except Exception as e:
        return False

print("Devices accepting 24kHz:")
for d in out_devs:
    if ok_24k(d["index"], 1):
        print(f"  [{d['index']:>2}] {d['name']}")


Python: /home/lonce/micromamba/envs/rnencodec/bin/python
PortAudio: (1246720, 'PortAudio V19.6.0-devel, revision 396fe4b6699ae929d3a685b3ef8a7e97396139a4')
Host APIs: ['ALSA', 'OSS']
Output devices (11):
  [ 0] HDA NVidia: HDMI 0 (hw:0,3)  (hostapi=ALSA)  max_out=8
  [ 1] HDA NVidia: HDMI 1 (hw:0,7)  (hostapi=ALSA)  max_out=8
  [ 2] HDA NVidia: HDMI 2 (hw:0,8)  (hostapi=ALSA)  max_out=8
  [ 3] HDA NVidia: HDMI 3 (hw:0,9)  (hostapi=ALSA)  max_out=8
  [ 4] HDA Intel PCH: ALC897 Analog (hw:1,0)  (hostapi=ALSA)  max_out=2
  [ 5] HDA Intel PCH: ALC897 Digital (hw:1,1)  (hostapi=ALSA)  max_out=2
  [ 7] HDA Intel PCH: HDMI 0 (hw:1,3)  (hostapi=ALSA)  max_out=8
  [ 8] HDA Intel PCH: HDMI 1 (hw:1,7)  (hostapi=ALSA)  max_out=8
  [ 9] HDA Intel PCH: HDMI 2 (hw:1,8)  (hostapi=ALSA)  max_out=8
  [10] HDA Intel PCH: HDMI 3 (hw:1,9)  (hostapi=ALSA)  max_out=8
  [11] hdmi  (hostapi=ALSA)  max_out=8
Default device: [4, 0]
Devices accepting 24kHz:


In [3]:
import sounddevice as sd, numpy as np

PREFERRED_NAMES = ("pulse", "pipewire", "default", "sysdefault")  # friendliest
picked = None
for d in sd.query_devices():
    if d["max_output_channels"] < 1: 
        continue
    name = d["name"].lower()
    if any(k in name for k in PREFERRED_NAMES) and \
       sd.check_output_settings(device=d["index"], samplerate=24000, channels=1) is None:
        picked = d; break

# If none of the friendly names worked, fall back to the first device that accepts 24k
if not picked:
    for d in sd.query_devices():
        if d["max_output_channels"]>0:
            try:
                sd.check_output_settings(device=d["index"], samplerate=24000, channels=1)
                picked = d; break
            except Exception:
                pass

if not picked:
    raise RuntimeError("No output device accepts 24 kHz in this env.")

sd.default.device = picked["index"]
sd.default.samplerate = 24000
sd.default.channels = 1

# sanity tone
t = np.linspace(0, 0.5, 12000, endpoint=False)
tone = (0.2*np.sin(2*np.pi*1000*t)).astype("float32")
sd.play(tone, 24000); sd.wait()
print("Using:", picked["index"], picked["name"])


RuntimeError: No output device accepts 24 kHz in this env.

In [4]:
import soxr
class Up2x48kStream:
    """Feed N at 24 kHz → get exactly 2N at 48 kHz every call (pads during startup)."""
    def __init__(self, channels=1, dtype="float32", quality="HQ"):
        self.ch = int(channels)
        self.dtype = dtype
        self.rs = soxr.ResampleStream(24000, 48000, num_channels=self.ch, dtype=dtype, quality=quality)
        self.buf = np.zeros((0, self.ch), dtype=dtype) if self.ch > 1 else np.zeros(0, dtype=dtype)

    def process(self, y24):
        x = np.asarray(y24, dtype=self.dtype, order="C")
        if self.ch > 1 and x.ndim == 1:
            x = np.tile(x[:, None], (1, self.ch))
        y48_new = self.rs.resample_chunk(x)                         # stateful
        # append to queue
        self.buf = (np.concatenate([self.buf, y48_new], axis=0) if self.ch > 1
                    else np.concatenate([self.buf, y48_new], axis=0))
        want = (x.shape[0] * 2)
        # pad during initial latency so we always return exactly 2N
        if self.buf.shape[0] < want:
            deficit = want - self.buf.shape[0]
            pad = (np.zeros((deficit, self.ch), dtype=self.dtype) if self.ch > 1
                   else np.zeros(deficit, dtype=self.dtype))
            out = (np.concatenate([self.buf, pad], axis=0))
            self.buf = self.buf[0:0]
            return out
        out = self.buf[:want]
        self.buf = self.buf[want:]
        return out

foo=Up2x48kStream()
x=np.random.uniform(-1, 1, 320)
y=foo.process(x)

In [5]:
foo=Up2x48kStream()
x=np.random.uniform(-1, 1, 320)
y=foo.process(x)

In [6]:
len(y)

640

In [7]:
y

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [8]:
import ctypes; print(ctypes.CDLL("libasound.so.2")._name)

libasound.so.2


In [9]:
import sounddevice as sd

def pick_output(sr=48000):
    devs = sd.query_devices()
    names = [d["name"] for d in devs]
    # Prefer pulse/pipewire/default if present
    for key in ("default", "pulse", "pipewire"):
        for i, d in enumerate(devs):
            if key in d["name"].lower() and d["max_output_channels"] > 0:
                try:
                    sd.check_output_settings(device=i, samplerate=sr, channels=1)
                    return i
                except Exception:
                    pass
    # Otherwise pick a non-HDMI analog device that accepts 48 kHz
    for i, d in enumerate(devs):
        name = d["name"].lower()
        if d["max_output_channels"] > 0 and "hdmi" not in name:
            try:
                sd.check_output_settings(device=i, samplerate=sr, channels=1)
                return i
            except Exception:
                continue
    raise RuntimeError("No suitable output device found")

device_index = pick_output(48000)
sd.default.device = (sd.default.device[0], device_index)  # keep current input; set output
sd.default.samplerate = 48000
sd.default.channels = 1